In [ ]:
import time
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit

# -----------------------------
# Data setup and group-aware split
# -----------------------------
file_name = 'FSS_DATA.csv'
input_columns = ['P', 'D', 'W', 'G', 'F']
output_columns = ['S11', 'S21']
group_column = 'ID'

df = pd.read_csv(file_name)

X = df[input_columns]
y = df[output_columns]
groups = df[group_column]

# First split: train/test
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train_raw = X.iloc[train_idx]
y_train_raw = y.iloc[train_idx]
X_test_raw = X.iloc[test_idx]
y_test_raw = y.iloc[test_idx]
groups_train = groups.iloc[train_idx]

# Second split: train/val inside train
# 10% of the remaining 80% becomes validation 
gss_val = GroupShuffleSplit(n_splits=1, test_size=0.1, random_state=42)
train_sub_idx, val_idx = next(gss_val.split(X_train_raw, y_train_raw, groups=groups_train))

X_val_raw = X_train_raw.iloc[val_idx]
y_val_raw = y_train_raw.iloc[val_idx]
X_train_raw = X_train_raw.iloc[train_sub_idx]
y_train_raw = y_train_raw.iloc[train_sub_idx]

# Scale with train-only fit
scaler_X = StandardScaler()
X_train = scaler_X.fit_transform(X_train_raw)
X_val = scaler_X.transform(X_val_raw)
X_test = scaler_X.transform(X_test_raw)

scaler_y = StandardScaler()
y_train = scaler_y.fit_transform(y_train_raw)
y_val = scaler_y.transform(y_val_raw)
y_test = scaler_y.transform(y_test_raw)

print('Data loaded and split successfully.')
print(f'Train samples: {X_train.shape[0]}')
print(f'Val samples  : {X_val.shape[0]}')
print(f'Test samples : {X_test.shape[0]}')
print(f'Input dim    : {X_train.shape[1]}')
print(f'Target dim   : {y_train.shape[1]}')
print(f'Input columns: {input_columns}')
print(f'Output columns: {output_columns}')

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# -----------------------------
# Model definition
# -----------------------------
def build_ann(input_dim):
    model = Sequential([
        Dense(192, activation='relu', input_shape=(input_dim,)),
        Dense(96, activation='relu'),
        Dense(48, activation='relu'),
        Dense(24, activation='relu'),
        Dense(2, activation='linear')
    ])
    optimizer = tf.keras.optimizers.Adam(
        learning_rate=0.0008,
        beta_1=0.9,
        beta_2=0.999,
        epsilon=1e-7
    )
    model.compile(optimizer=optimizer, loss='mse')
    return model


def extract_lr_series(history_obj):
    if 'learning_rate' in history_obj.history:
        return history_obj.history['learning_rate']
    if 'lr' in history_obj.history:
        return history_obj.history['lr']
    return [np.nan] * len(history_obj.history['loss'])


# -----------------------------
# Multi-seed controlled comparison
# -----------------------------
seed_list = [123]
run_outputs = []
epoch_logs = []
time_rows = []

for seed in seed_list:
    print(f'\n===== Running seed {seed} =====')

    tf.keras.backend.clear_session()
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)

    base_model = build_ann(X_train.shape[1])
    initial_weights = base_model.get_weights()

    # -----------------------------
    # Strategy A: EarlyStopping + ReduceLR
    # -----------------------------
    model_es = build_ann(X_train.shape[1])
    model_es.set_weights(initial_weights)

    early_stop = EarlyStopping(
        monitor='val_loss',
        patience=35,
        min_delta=1e-4,
        restore_best_weights=True,
        start_from_epoch=30
    )

    reduce_lr_es = ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=10,
        cooldown=3,
        min_lr=1e-6,
        min_delta=1e-4
    )

    start_es = time.perf_counter()
    history_es = model_es.fit(
        X_train, y_train,
        epochs=200,
        batch_size=512,
        validation_data=(X_val, y_val),
        verbose=1,
        callbacks=[early_stop, reduce_lr_es]
    )
    time_es = time.perf_counter() - start_es

    # -----------------------------
    # Strategy B: Plain training
    # -----------------------------
    model_plain = build_ann(X_train.shape[1])
    model_plain.set_weights(initial_weights)

    start_plain = time.perf_counter()
    history_plain = model_plain.fit(
        X_train, y_train,
        epochs=200,
        batch_size=512,
        validation_data=(X_val, y_val),
        verbose=1,
        callbacks=[]
    )
    time_plain = time.perf_counter() - start_plain

    pred_es = model_es.predict(X_test, verbose=1)
    pred_plain = model_plain.predict(X_test, verbose=1)

    epochs_es = len(history_es.history['loss'])
    epochs_plain = len(history_plain.history['loss'])

    run_outputs.append({
        'Seed': seed,
        'model_es': model_es,
        'model_plain': model_plain,
        'history_es': history_es,
        'history_plain': history_plain,
        'pred_es': pred_es,
        'pred_plain': pred_plain,
        'epochs_es': epochs_es,
        'epochs_plain': epochs_plain,
        'time_es': time_es,
        'time_plain': time_plain
    })

    time_rows.append({
        'Seed': seed,
        'Time_ES_sec': time_es,
        'Time_Plain_sec': time_plain,
        'Saved_sec': time_plain - time_es
    })

    lr_es = extract_lr_series(history_es)
    lr_plain = extract_lr_series(history_plain)

    for i, (tr, va, lr) in enumerate(zip(history_es.history['loss'], history_es.history['val_loss'], lr_es), start=1):
        epoch_logs.append({
            'Seed': seed,
            'Strategy': 'ES+ReduceLR',
            'Epoch': i,
            'TrainLoss': tr,
            'ValLoss': va,
            'LearningRate': lr
        })

    for i, (tr, va, lr) in enumerate(zip(history_plain.history['loss'], history_plain.history['val_loss'], lr_plain), start=1):
        epoch_logs.append({
            'Seed': seed,
            'Strategy': 'Plain',
            'Epoch': i,
            'TrainLoss': tr,
            'ValLoss': va,
            'LearningRate': lr
        })

df_epoch_logs = pd.DataFrame(epoch_logs)
df_time = pd.DataFrame(time_rows)

print('\nTime comparison table:')
print(df_time.round(4).to_string(index=False))

print('\nPer-epoch log sample:')
print(df_epoch_logs.head(10).to_string(index=False))


In [ ]:
# -----------------------------
# Visualization: loss curves and training times
# -----------------------------
for run in run_outputs:
    seed = run['Seed']
    history_es = run['history_es']
    history_plain = run['history_plain']

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)

    # Left: ES + ReduceLR
    loss_es = history_es.history['loss']
    val_es = history_es.history['val_loss']
    epochs_es = np.arange(1, len(loss_es) + 1)
    best_es_idx = int(np.argmin(val_es))

    axes[0].plot(epochs_es, loss_es, label='Train Loss', linewidth=2)
    axes[0].plot(epochs_es, val_es, label='Val Loss', linewidth=2)
    axes[0].scatter(best_es_idx + 1, val_es[best_es_idx], color='red', zorder=3)
    axes[0].annotate(
        f'Best: ep {best_es_idx + 1}\n{val_es[best_es_idx]:.4g}',
        xy=(best_es_idx + 1, val_es[best_es_idx]),
        xytext=(best_es_idx + 1, val_es[best_es_idx] * 1.05 if val_es[best_es_idx] != 0 else 0.01),
        arrowprops=dict(arrowstyle='->', lw=1)
    )
    axes[0].set_title(f'Seed {seed} - ES+ReduceLR')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss (MSE)')
    axes[0].grid(True, linestyle='--', alpha=0.5)
    axes[0].legend()

    # Right: Plain training
    loss_plain = history_plain.history['loss']
    val_plain = history_plain.history['val_loss']
    epochs_plain = np.arange(1, len(loss_plain) + 1)
    best_plain_idx = int(np.argmin(val_plain))

    axes[1].plot(epochs_plain, loss_plain, label='Train Loss', linewidth=2)
    axes[1].plot(epochs_plain, val_plain, label='Val Loss', linewidth=2)
    axes[1].scatter(best_plain_idx + 1, val_plain[best_plain_idx], color='red', zorder=3)
    axes[1].annotate(
        f'Best: ep {best_plain_idx + 1}\n{val_plain[best_plain_idx]:.4g}',
        xy=(best_plain_idx + 1, val_plain[best_plain_idx]),
        xytext=(best_plain_idx + 1, val_plain[best_plain_idx] * 1.05 if val_plain[best_plain_idx] != 0 else 0.01),
        arrowprops=dict(arrowstyle='->', lw=1)
    )
    axes[1].set_title(f'Seed {seed} - Plain')
    axes[1].set_xlabel('Epoch')
    axes[1].grid(True, linestyle='--', alpha=0.5)
    axes[1].legend()

    plt.tight_layout()
    plt.show()

# Per-seed ES vs Plain training-time bars
x = np.arange(len(df_time))
width = 0.35

plt.figure(figsize=(8, 4.5))
plt.bar(x - width / 2, df_time['Time_ES_sec'], width=width, label='ES+ReduceLR')
plt.bar(x + width / 2, df_time['Time_Plain_sec'], width=width, label='Plain')
plt.xticks(x, [f'Seed {s}' for s in df_time['Seed']])
plt.ylabel('Time (sec)')
plt.title('Training Time per Seed')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()

avg_es = df_time['Time_ES_sec'].mean()
avg_plain = df_time['Time_Plain_sec'].mean()

plt.figure(figsize=(6, 4.5))
plt.bar(['ES+ReduceLR', 'Plain'], [avg_es, avg_plain], width=0.5)
plt.ylabel('Average Time (sec)')
plt.title('Average Training Time by Strategy')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, explained_variance_score

# -----------------------------
# Evaluation on original target scale
# -----------------------------
def evaluate_outputs_on_original_scale(y_true_scaled, y_pred_scaled):
    y_true_orig = scaler_y.inverse_transform(np.asarray(y_true_scaled))
    y_pred_orig = scaler_y.inverse_transform(np.asarray(y_pred_scaled))

    rows = []
    for j, output_name in enumerate(output_columns):
        y_true = y_true_orig[:, j]
        y_pred = y_pred_orig[:, j]
        mse = mean_squared_error(y_true, y_pred)

        rows.append({
            'Output': output_name,
            'MAE': mean_absolute_error(y_true, y_pred),
            'MSE': mse,
            'RMSE': np.sqrt(mse),
            'R2': r2_score(y_true, y_pred),
            'Variance': explained_variance_score(y_true, y_pred)
        })

    return pd.DataFrame(rows)

rows = []
for run in run_outputs:
    seed = run['Seed']

    metrics_es = evaluate_outputs_on_original_scale(y_test, run['pred_es'])
    metrics_es['Seed'] = seed
    metrics_es['Strategy'] = 'ES+ReduceLR'
    metrics_es['Epochs'] = run['epochs_es']
    rows.append(metrics_es)

    metrics_plain = evaluate_outputs_on_original_scale(y_test, run['pred_plain'])
    metrics_plain['Seed'] = seed
    metrics_plain['Strategy'] = 'Plain'
    metrics_plain['Epochs'] = run['epochs_plain']
    rows.append(metrics_plain)

df_runs = pd.concat(rows, ignore_index=True)
df_runs = df_runs[['Seed', 'Output', 'Strategy', 'Epochs', 'MAE', 'MSE', 'RMSE', 'R2', 'Variance']]

print('Per-run metrics on original scale:')
print(df_runs.round(6).sort_values(['Output', 'Seed', 'Strategy']).to_string(index=False))

summary = (
    df_runs
    .groupby(['Output', 'Strategy'])[['MAE', 'MSE', 'RMSE', 'R2', 'Variance', 'Epochs']]
    .agg(['mean', 'std'])
    .round(6)
)
print('\nGrouped summary (mean/std by output and strategy):')
print(summary)

# Count per-seed metric wins separately for S11 and S21
metrics_lower_better = ['MAE', 'MSE', 'RMSE']
metrics_higher_better = ['R2', 'Variance']

seed_win_rows = []
for output_name in output_columns:
    for seed in sorted(df_runs['Seed'].unique()):
        sub = (
            df_runs[
                (df_runs['Output'] == output_name) &
                (df_runs['Seed'] == seed)
            ]
            .set_index('Strategy')
        )

        wins_es = 0
        for metric in metrics_lower_better:
            if sub.loc['ES+ReduceLR', metric] < sub.loc['Plain', metric]:
                wins_es += 1
        for metric in metrics_higher_better:
            if sub.loc['ES+ReduceLR', metric] > sub.loc['Plain', metric]:
                wins_es += 1

        seed_win_rows.append({
            'Output': output_name,
            'Seed': seed,
            'ES_Wins_Among_5': wins_es,
            'ES_Better_At_Least_3of5': wins_es >= 3
        })

df_seed_wins = pd.DataFrame(seed_win_rows)
print('\nPer-seed ES+ReduceLR win count (3/5 rule):')
print(df_seed_wins.to_string(index=False))
